In [1]:
# 代码作用：读取MES、LIMS数据汇总为 002炼糖期分蜜指标统计报表v1
# 核心逻辑：全局累计合格率（按月度、班次独立累计）
# 2026/04/02
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType, DateType,DoubleType
# 关键：明确导入Spark函数，避免和Python内置函数混淆
from pyspark.sql.functions import (
    current_timestamp, split, lit, col, row_number, sum as spark_sum, 
    round as spark_round, avg, when, to_date, count, coalesce
)
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()



# ===================== 1. 读取LIMS样品分析 R3糖 数据 =====================
lims_r3_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_order_no,
        test_name1,                     -- 补上逗号
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '炼糖生产'
        AND ord_sample_name LIKE '%R3糖浆%'                     
        AND test_name1 IN ('锤度', '色值')                          
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_order_no,
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS r3_brix,
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS r3_color_value
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_order_no
HAVING MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) IS NOT NULL 
    OR MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL;
"""
lims_r3_df = spark.sql(lims_r3_sql)

# ===================== 2. 读取LIMS样品分析 炼糖桔水 数据 =====================
lims_js_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        test_name1,
        ord_up_ver,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '炼糖生产'
        AND ord_sample_name LIKE '%炼糖桔水%'
        AND test_name1 IN ('锤度', '重力纯度', '产量')
),
paste_b_data AS (
    SELECT 
        company,
        crushing_season,
        record_date AS paste_record_date,
        work_shift AS paste_work_shift,
        MIN(ord_up_ver) AS min_ord_up_ver,
        MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS molasses_brix,
        MAX(CASE WHEN test_name1 = '重力纯度' THEN test_value END) AS gravity_purity,
        MAX(CASE WHEN test_name1 = '产量' THEN test_value END) AS yield
    FROM raw_data
    WHERE ord_sample_name LIKE '%炼糖桔水%'
    GROUP BY company, crushing_season, record_date, work_shift
),
ranked_paste_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY company, crushing_season, paste_record_date, paste_work_shift
            ORDER BY min_ord_up_ver ASC
        ) AS rn
    FROM paste_b_data
)
SELECT 
    company,
    crushing_season,
    paste_record_date,
    paste_work_shift,
    molasses_brix,
    gravity_purity,
    yield
FROM ranked_paste_data
WHERE rn = 1
  AND (
        (CASE WHEN molasses_brix IS NOT NULL THEN 1 ELSE 0 END) +
        (CASE WHEN gravity_purity IS NOT NULL THEN 1 ELSE 0 END) +
        (CASE WHEN yield IS NOT NULL THEN 1 ELSE 0 END)
      ) >= 2
"""
lims_js_df = spark.sql(lims_js_sql)

# ===================== 3. 读取LIMS样品分析 洄溶糖 数据 =====================
lims_hrt_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '炼糖生产'
        AND ord_sample_name = '洄溶糖'
        AND test_name1 = '重量'
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '重量' THEN test_value END) AS remelt_syrup_yield
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING MAX(CASE WHEN test_name1 = '重量' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC
"""
lims_hrt_df = spark.sql(lims_hrt_sql)

# ===================== 3. 读取LIMS 精糖指标合格标准数据 =====================
lims_jtzb_sql = """
SELECT 
*
FROM dwd.dwd_lims_sugar_quality_standard_f_1h
WHERE org_code = 'FNR'
  AND production_category = '炼糖生产'
  AND material_name IN ('R3糖浆','炼糖桔水')
  AND dismonth_date = (
      SELECT MAX(dismonth_date)
      FROM dwd.dwd_lims_sugar_quality_standard_f_1h
      WHERE org_code = 'FNR'
        AND production_category = '炼糖生产'
        AND material_name IN ('R3糖浆','炼糖桔水')
  )
;
"""
lims_jtzb_df = spark.sql(lims_jtzb_sql)
lims_jtzb_df.show()

Spark 启动中...
+--------+-------------------+---------------+----------+-------------+---------+-----------+-----------+--------------+-------------------+-------------------+----------+-------------+
|org_code|production_category|crushing_season|  dismonth|material_name|item_name|lower_limit|upper_limit|standard_value|        create_time|        update_time|created_by|dismonth_date|
+--------+-------------------+---------------+----------+-------------+---------+-----------+-----------+--------------+-------------------+-------------------+----------+-------------+
|     FNR|           炼糖生产|          24/25|2025年10月|     炼糖桔水|   视纯度|       0.00|      55.00|           ≤55|2025-09-29 10:53:02|2025-09-29 10:58:51|    韦愉谢|   2025-10-01|
|     FNR|           炼糖生产|          24/25|2025年10月|     炼糖桔水|     锤度|      86.00|      90.00|        86～90|2025-09-29 10:53:02|2025-09-29 10:58:51|    韦愉谢|   2025-10-01|
|     FNR|           炼糖生产|          24/25|2025年10月|       R3糖浆|     锤度|      58.00|      

In [2]:
lims_hrt_df.show()

+-------+---------------+-----------+----------+---------------+--------------+------------------+
|company|crushing_season|record_date|work_shift|ord_sample_name|        ord_id|remelt_syrup_yield|
+-------+---------------+-----------+----------+---------------+--------------+------------------+
|    FNR|          25/26| 2026-04-07|      乙班|         洄溶糖|81317323815808|            220.90|
|    FNR|          25/26| 2026-04-07|      丁班|         洄溶糖|81354057563008|            222.44|
|    FNR|          25/26| 2026-04-07|      甲班|         洄溶糖|81295977102208|            221.40|
|    FNR|          25/26| 2026-04-06|      甲班|         洄溶糖|81208082676608|            223.05|
|    FNR|          25/26| 2026-04-06|      丙班|         洄溶糖|81266407259008|            223.07|
|    FNR|          25/26| 2026-04-06|      乙班|         洄溶糖|81237435022208|            221.45|
|    FNR|          25/26| 2026-04-05|      丙班|         洄溶糖|81177932839808|            239.63|
|    FNR|          25/26| 2026-04-05|      乙班

In [3]:
lims_js_df.show()

+-------+---------------+-----------------+----------------+-------------+--------------+-----+
|company|crushing_season|paste_record_date|paste_work_shift|molasses_brix|gravity_purity|yield|
+-------+---------------+-----------------+----------------+-------------+--------------+-----+
|    FNR|          22/23|       2023-03-04|            丁班|        86.50|         55.95|57.48|
|    FNR|          22/23|       2023-03-05|            甲班|        86.50|         55.95|30.00|
|    FNR|          22/23|       2023-03-06|            甲班|        88.56|         55.92|30.63|
|    FNR|          22/23|       2023-03-07|            丙班|        88.56|         55.92|33.21|
|    FNR|          22/23|       2023-03-08|            乙班|        88.56|         55.92|20.00|
|    FNR|          22/23|       2023-03-09|            乙班|        88.56|         55.92|30.00|
|    FNR|          22/23|       2023-03-09|            甲班|        88.94|         55.19| 0.00|
|    FNR|          22/23|       2023-03-10|           

In [5]:
# ===================== 完整代码：计算所有指标并写入 Hive 表（字段英文小写） =====================
# 前提：您的 SparkSession 已经启用 Hive 支持（.enableHiveSupport()）

from pyspark.sql.functions import month, col, when, count, sum as spark_sum, round as spark_round, lit, concat
from pyspark.sql.functions import when as when_

print("开始计算所有指标（值不乘100，月份为“3月”“4月”等）...")

# -------------------------------------------------------------------
# 辅助函数：统一班次、日期字段名
# -------------------------------------------------------------------
def normalize_shift_date(df, shift_col, date_col):
    return df.withColumnRenamed(shift_col, "work_shift") \
             .withColumnRenamed(date_col, "record_date")

js_df = normalize_shift_date(lims_js_df, "paste_work_shift", "paste_record_date")
hrt_df = lims_hrt_df

# -------------------------------------------------------------------
# 通用合格率计算函数（值不乘100，月份为“X月”）
# -------------------------------------------------------------------
def compute_metrics(df, value_col, indicator_name):
    # 生成月份字段：数字月份 + "月"，并保留数字月份用于排序
    df_with_month = df.withColumn("month_num", month("record_date")) \
                      .withColumn("month_disp", concat(col("month_num"), lit("月")))
    
    # 班次月度
    month_res = df_with_month.groupBy("company", "crushing_season", "work_shift", "month_disp", "month_num") \
        .agg(spark_sum("is_qualified").alias("qualified_cnt"), count(value_col).alias("total_cnt")) \
        .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
        .select("company", "crushing_season", 
                lit(indicator_name).alias("indicator"), 
                col("work_shift").alias("shift"), 
                col("month_disp").alias("month"), 
                "value", "month_num")
    # 班次累计
    cum_res = df.groupBy("company", "crushing_season", "work_shift") \
        .agg(spark_sum("is_qualified").alias("qualified_cnt"), count(value_col).alias("total_cnt")) \
        .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
        .select("company", "crushing_season", 
                lit(indicator_name).alias("indicator"), 
                col("work_shift").alias("shift"), 
                lit("累计").alias("month"), 
                "value") \
        .withColumn("month_num", lit(999))
    # 工段月度
    dept_month = df_with_month.groupBy("company", "crushing_season", "month_disp", "month_num") \
        .agg(spark_sum("is_qualified").alias("qualified_cnt"), count(value_col).alias("total_cnt")) \
        .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
        .select("company", "crushing_season", 
                lit(indicator_name).alias("indicator"), 
                lit("工段合计").alias("shift"), 
                col("month_disp").alias("month"), 
                "value", "month_num")
    # 工段累计
    dept_cum = df.groupBy("company", "crushing_season") \
        .agg(spark_sum("is_qualified").alias("qualified_cnt"), count(value_col).alias("total_cnt")) \
        .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
        .select("company", "crushing_season", 
                lit(indicator_name).alias("indicator"), 
                lit("工段合计").alias("shift"), 
                lit("累计").alias("month"), 
                "value") \
        .withColumn("month_num", lit(999))
    return month_res.union(cum_res).union(dept_month).union(dept_cum)

# -------------------------------------------------------------------
# 1. R3糖浆锤度合格率(%)
# -------------------------------------------------------------------
brix_std = lims_jtzb_df.filter(
    (col("material_name") == "R3糖浆") & (col("item_name") == "锤度")
).select("lower_limit", "upper_limit").first()
BRIX_LOWER, BRIX_UPPER = (brix_std["lower_limit"], brix_std["upper_limit"]) if brix_std else (58.0, 63.0)
print(f"R3锤度标准: {BRIX_LOWER} ~ {BRIX_UPPER}")

r3_brix = lims_r3_df.filter(col("r3_brix").isNotNull()) \
    .withColumn("is_qualified", when((col("r3_brix") >= BRIX_LOWER) & (col("r3_brix") <= BRIX_UPPER), 1).otherwise(0))
r3_brix_res = compute_metrics(r3_brix, "r3_brix", "R3糖浆锤度合格率(%)")

# -------------------------------------------------------------------
# 2. R3糖浆色值合格率(%)
# -------------------------------------------------------------------
color_std = lims_jtzb_df.filter(
    (col("material_name") == "R3糖浆") & (col("item_name") == "色值")
).select("upper_limit").first()
COLOR_UPPER = color_std["upper_limit"] if color_std and color_std["upper_limit"] is not None else 500.0
print(f"R3色值标准: ≤ {COLOR_UPPER}")

r3_color = lims_r3_df.filter(col("r3_color_value").isNotNull()) \
    .withColumn("is_qualified", when(col("r3_color_value") <= COLOR_UPPER, 1).otherwise(0))
r3_color_res = compute_metrics(r3_color, "r3_color_value", "R3糖浆色值合格率(%)")

# -------------------------------------------------------------------
# 3. 桔水锤度
# -------------------------------------------------------------------
js_brix_std = lims_jtzb_df.filter(
    (col("material_name") == "炼糖桔水") & (col("item_name") == "锤度")
).select("lower_limit", "upper_limit").first()
JS_BRIX_LOWER, JS_BRIX_UPPER = (js_brix_std["lower_limit"], js_brix_std["upper_limit"]) if js_brix_std else (86.0, 90.0)
print(f"桔水锤度标准: {JS_BRIX_LOWER} ~ {JS_BRIX_UPPER}")

js_brix = js_df.filter(col("molasses_brix").isNotNull()) \
    .withColumn("is_qualified", when((col("molasses_brix") >= JS_BRIX_LOWER) & (col("molasses_brix") <= JS_BRIX_UPPER), 1).otherwise(0))
js_brix_res = compute_metrics(js_brix, "molasses_brix", "桔水锤度合格率(%)")

# -------------------------------------------------------------------
# 4. 桔水重力纯度
# -------------------------------------------------------------------
js_purity_std = lims_jtzb_df.filter(
    (col("material_name") == "炼糖桔水") & (col("item_name") == "重力纯度")
).select("upper_limit").first()
JS_PURITY_UPPER = js_purity_std["upper_limit"] if js_purity_std and js_purity_std["upper_limit"] is not None else 57.0
print(f"桔水重力纯度标准: ≤ {JS_PURITY_UPPER}")

js_purity = js_df.filter(col("gravity_purity").isNotNull()) \
    .withColumn("is_qualified", when(col("gravity_purity") <= JS_PURITY_UPPER, 1).otherwise(0))
js_purity_res = compute_metrics(js_purity, "gravity_purity", "桔水重力纯度合格率(%)")

# -------------------------------------------------------------------
# 5. 废糖蜜产率(%)（左连接：洄溶糖为主表，桔水缺失时 yield 填 0）
# -------------------------------------------------------------------
# 桔水数据：按公司、榨季、日期、班次聚合 yield（求和）
js_agg = js_df.select("company", "crushing_season", "record_date", "work_shift", "yield") \
    .groupBy("company", "crushing_season", "record_date", "work_shift") \
    .agg(spark_sum("yield").alias("yield"))

# 洄溶糖数据：按公司、榨季、日期、班次聚合 remelt_syrup_yield（求和）
hrt_agg = hrt_df.select("company", "crushing_season", "record_date", "work_shift", "remelt_syrup_yield") \
    .groupBy("company", "crushing_season", "record_date", "work_shift") \
    .agg(spark_sum("remelt_syrup_yield").alias("remelt_syrup_yield"))

# 左连接：以洄溶糖为主表，桔水数据缺失时 yield 填 0
joined = hrt_agg.join(js_agg, on=["company", "crushing_season", "record_date", "work_shift"], how="left") \
    .fillna({"yield": 0})   # 连接不上的桔水 yield 填 0

# 过滤掉洄溶糖产量为空或为0的记录（产率无意义，但保留记录？根据业务决定）
# 注意：若需要保留所有洄溶糖班次（即使 remelt=0），则应保留并设置产率为0
# 以下采用保留所有班次，产率为0的逻辑（不进行 filter）
# 但需要避免除零，在计算时用 when 处理
joined = joined.withColumn("month_num", month("record_date")) \
               .withColumn("month_disp", concat(col("month_num"), lit("月")))

# 班次月度
monthly_agg = joined.groupBy("company", "crushing_season", "work_shift", "month_disp", "month_num") \
    .agg(spark_sum("yield").alias("sum_yield"), spark_sum("remelt_syrup_yield").alias("sum_remelt")) \
    .withColumn("value", 
                when(col("sum_remelt") == 0, lit(0.0))
                .otherwise(spark_round(col("sum_yield") / col("sum_remelt"), 4))) \
    .select("company", "crushing_season", 
            lit("废糖蜜产率(%)").alias("indicator"), 
            col("work_shift").alias("shift"), 
            col("month_disp").alias("month"), 
            "value", "month_num")

# 班次累计
cum_agg = joined.groupBy("company", "crushing_season", "work_shift") \
    .agg(spark_sum("yield").alias("sum_yield"), spark_sum("remelt_syrup_yield").alias("sum_remelt")) \
    .withColumn("value", 
                when(col("sum_remelt") == 0, lit(0.0))
                .otherwise(spark_round(col("sum_yield") / col("sum_remelt"), 4))) \
    .select("company", "crushing_season", 
            lit("废糖蜜产率(%)").alias("indicator"), 
            col("work_shift").alias("shift"), 
            lit("累计").alias("month"), 
            "value") \
    .withColumn("month_num", lit(999))

# 工段月度
dept_monthly = joined.groupBy("company", "crushing_season", "month_disp", "month_num") \
    .agg(spark_sum("yield").alias("sum_yield"), spark_sum("remelt_syrup_yield").alias("sum_remelt")) \
    .withColumn("value", 
                when(col("sum_remelt") == 0, lit(0.0))
                .otherwise(spark_round(col("sum_yield") / col("sum_remelt"), 4))) \
    .select("company", "crushing_season", 
            lit("废糖蜜产率(%)").alias("indicator"), 
            lit("工段合计").alias("shift"), 
            col("month_disp").alias("month"), 
            "value", "month_num")

# 工段累计
dept_cum = joined.groupBy("company", "crushing_season") \
    .agg(spark_sum("yield").alias("sum_yield"), spark_sum("remelt_syrup_yield").alias("sum_remelt")) \
    .withColumn("value", 
                when(col("sum_remelt") == 0, lit(0.0))
                .otherwise(spark_round(col("sum_yield") / col("sum_remelt"), 4))) \
    .select("company", "crushing_season", 
            lit("废糖蜜产率(%)").alias("indicator"), 
            lit("工段合计").alias("shift"), 
            lit("累计").alias("month"), 
            "value") \
    .withColumn("month_num", lit(999))

js_yield_res = monthly_agg.union(cum_agg).union(dept_monthly).union(dept_cum)

# -------------------------------------------------------------------
# 6. 合并所有指标并排序（排序仅用于展示，写入表时不需要保留排序列）
# -------------------------------------------------------------------
final_df = r3_brix_res.union(r3_color_res) \
                      .union(js_brix_res) \
                      .union(js_purity_res) \
                      .union(js_yield_res)

# 添加排序字段（仅用于 show，不影响写入）
sorted_df = final_df.withColumn("shift_order",
    when_(col("shift") == "甲班", 1)
    .when(col("shift") == "乙班", 2)
    .when(col("shift") == "丙班", 3)
    .when(col("shift") == "丁班", 4)
    .when(col("shift") == "工段合计", 5).otherwise(6)
).withColumn("indicator_order",
    when_(col("indicator") == "R3糖浆锤度合格率(%)", 1)
    .when(col("indicator") == "R3糖浆色值合格率(%)", 2)
    .when(col("indicator") == "桔水锤度合格率(%)", 3)
    .when(col("indicator") == "桔水重力纯度合格率(%)", 4)
    .when(col("indicator") == "废糖蜜产率(%)", 5).otherwise(6)
).orderBy("company", "crushing_season", "indicator_order", "shift_order", "month_num") \
 .drop("shift_order", "indicator_order", "month_num")

print("所有指标计算完成（值均为小数，月份为“3月”“4月”等）:")
sorted_df.show(500, truncate=False)

# ===================== 写入 Hive 表 =====================
table_name = "app.app_lims_refining_indicators_summary_f_1h"
print(f"正在写入表 {table_name} ...")

# 先删除旧表（可选，避免残留分区）
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# 写入数据（不分区）
sorted_df.write.mode("overwrite") \
    .format("parquet") \
    .saveAsTable(table_name)

print(f"数据已成功写入表 {table_name}")
print("作业完成。")

开始计算所有指标（值不乘100，月份为“3月”“4月”等）...
R3锤度标准: 58.00 ~ 63.00
R3色值标准: ≤ 500.00
桔水锤度标准: 86.00 ~ 90.00
桔水重力纯度标准: ≤ 57.00
所有指标计算完成（值均为小数，月份为“3月”“4月”等）:
+-------+---------------+---------------------+--------+-----+------+
|company|crushing_season|indicator            |shift   |month|value |
+-------+---------------+---------------------+--------+-----+------+
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |3月  |1.0   |
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |4月  |1.0   |
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |5月  |0.9677|
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |6月  |1.0   |
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |7月  |1.0   |
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |8月  |1.0   |
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |9月  |1.0   |
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |11月 |0.8261|
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |12月 |1.0   |
|FNR    |22/23          |R3糖浆锤度合格率(%)  |甲班    |累计 |0.9746|
|FNR    |22/23          |R3糖浆锤度合格率(%)  |乙班    |3月

数据已成功写入表 app.app_lims_refining_indicators_summary_f_1h
作业完成。
